In [7]:
import pandas as pd

eventos = pd.read_csv("../data/eventos_todos_completo.csv")
playbooks = pd.read_csv("../data/playbooks.csv")

print(eventos["indicador"].value_counts())
print(playbooks)

indicador
Payroll_EUA    43
CPI_EUA        39
IPCA_BR        34
Selic_BR       18
Name: count, dtype: int64
     indicador ativo_alvo direcao_se_surpresa_positiva  \
0      CPI_EUA        SPY                       vender   
1      IPCA_BR        EWZ                       vender   
2     Selic_BR      BRL=X                       vender   
3  Payroll_EUA        SPY                       vender   

  direcao_se_surpresa_negativa  
0                      comprar  
1                      comprar  
2                      comprar  
3                      comprar  


In [8]:
LIMIAR_SURPRESA = 1.0

eventos["limiar_ian_indicador"] = eventos.groupby("indicador")["IAN"].transform(lambda x: x.quantile(0.75))
eventos["opera"] = (eventos["surpresa_zscore"].abs() > LIMIAR_SURPRESA) & (eventos["IAN"] > eventos["limiar_ian_indicador"])

print(eventos.groupby("indicador")["opera"].sum())
print(f"\nTotal geral de operações: {eventos['opera'].sum()}")

indicador
CPI_EUA        5
IPCA_BR        4
Payroll_EUA    3
Selic_BR       2
Name: opera, dtype: int64

Total geral de operações: 14


In [9]:
def determinar_direcao(row, playbooks):
    if not row["opera"]:
        return None
    regra = playbooks[playbooks["indicador"] == row["indicador"]].iloc[0]
    if row["surpresa_zscore"] > 0:
        return regra["direcao_se_surpresa_positiva"]
    else:
        return regra["direcao_se_surpresa_negativa"]

eventos["direcao"] = eventos.apply(lambda row: determinar_direcao(row, playbooks), axis=1)

In [10]:
eventos["tamanho_posicao"] = eventos["IAN"] * (1 + eventos["ICE"])
eventos.loc[~eventos["opera"], "tamanho_posicao"] = 0

In [11]:
operacoes = eventos[eventos["opera"]]
print(operacoes[["data", "indicador", "surpresa_zscore", "IAN", "direcao", "tamanho_posicao"]])

           data    indicador  surpresa_zscore       IAN  direcao  \
1    2023-04-12      CPI_EUA        -1.007506  0.311111  comprar   
13   2024-04-10      CPI_EUA         1.007506  0.311111   vender   
33   2026-02-13      CPI_EUA        -1.007506  0.433333  comprar   
35   2026-04-10      CPI_EUA        -1.007506  0.333333  comprar   
38   2026-07-14      CPI_EUA        -3.022518  0.322222  comprar   
52   2024-02-01      IPCA_BR         1.570191  0.509434   vender   
63   2025-01-01      IPCA_BR        -1.237121  0.867925  comprar   
64   2025-02-01      IPCA_BR         4.182419  0.962264   vender   
65   2025-03-01      IPCA_BR         1.141957  0.641509   vender   
73   2023-08-03     Selic_BR        -2.402829  0.791045  comprar   
82   2024-12-12     Selic_BR         2.402829  0.820896   vender   
128  2026-03-06  Payroll_EUA        -1.745893  0.762887  comprar   
129  2026-04-03  Payroll_EUA         1.315240  0.618557   vender   
131  2026-06-05  Payroll_EUA         1.012618  1

In [12]:
eventos.to_csv("../data/eventos_com_decisao.csv", index=False)